# 📈 Notebook 03 — Exploratory Data Analysis & Visualization
**Owner:** Member 3

**Goal:** Visually explore the cleaned data to find patterns related to churn.

**Input:** Cleaned CSVs from `data/cleaned/`

**Output:** 8-10 key visualizations saved to `reports/figures/`

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Load cleaned datasets
renewal  = pd.read_csv('../data/cleaned/renewal_calls_clean.csv')
billing  = pd.read_csv('../data/cleaned/billings_clean.csv')
cc_calls = pd.read_csv('../data/cleaned/cc_calls_clean.csv')
emails   = pd.read_csv('../data/cleaned/emails_clean.csv')

print('All cleaned datasets loaded!')

## Plot 1: Churn Distribution
How many customers churned vs stayed?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Count churn categories
churn_counts = renewal['churn_category'].fillna('Not Churned').value_counts()
colors = ['#2ecc71' if x == 'Not Churned' else '#e74c3c' for x in churn_counts.index]

churn_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_title('Distribution of Churn Categories', fontsize=14, fontweight='bold')
ax.set_xlabel('Churn Category')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)

# Add count labels on bars
for i, (val, count) in enumerate(zip(churn_counts.index, churn_counts.values)):
    ax.text(i, count + 500, f'{count:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/01_churn_distribution.png', dpi=150)
plt.show()

## Plot 2: Call Direction Distribution
Are churning customers calling in (inbound) or being called (outbound)?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Create a churn flag for grouping
renewal['is_churned'] = renewal['churn_category'].notna().map({True: 'Churned', False: 'Retained'})

pd.crosstab(renewal['call_direction'], renewal['is_churned']).plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Call Direction: Churned vs Retained', fontsize=14, fontweight='bold')
ax.set_xlabel('Call Direction')
ax.set_ylabel('Number of Calls')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Status')

plt.tight_layout()
plt.savefig('../reports/figures/02_call_direction.png', dpi=150)
plt.show()

## Plot 3: Complaint Analysis
What are customers complaining about?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Serious complaints
renewal['serious_complaint'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%', 
                                                  colors=['#2ecc71', '#e74c3c', '#95a5a6'])
axes[0].set_title('Serious Complaints', fontweight='bold')
axes[0].set_ylabel('')

# Other complaints
renewal['other_complaint'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                                colors=['#2ecc71', '#e74c3c', '#95a5a6'])
axes[1].set_title('Other Complaints', fontweight='bold')
axes[1].set_ylabel('')

plt.suptitle('Complaint Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_complaints.png', dpi=150)
plt.show()

## Plot 4: Competitor Mentions
How often do customers mention competitors?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Competitor mention by churn status
pd.crosstab(renewal['explicit_competitor_mention'], renewal['is_churned']).plot(
    kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Competitor Mentions: Churned vs Retained', fontsize=14, fontweight='bold')
ax.set_xlabel('Competitor Mentioned?')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/figures/04_competitor_mentions.png', dpi=150)
plt.show()

## Plot 5: Desire to Cancel vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

pd.crosstab(renewal['desire_to_cancel'], renewal['is_churned']).plot(
    kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Desire to Cancel vs Churn Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Desire to Cancel')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/05_desire_to_cancel.png', dpi=150)
plt.show()

## Plot 6: Impact of Discounts on Churn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

pd.crosstab(renewal['discount_offered'], renewal['is_churned']).plot(
    kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Discount Offered vs Churn Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Was Discount Offered?')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/figures/06_discount_impact.png', dpi=150)
plt.show()

## Plot 7: Price Increase Discussion vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

pd.crosstab(renewal['discussion_on_price_increase'], renewal['is_churned']).plot(
    kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Price Increase Discussion vs Churn', fontsize=14, fontweight='bold')
ax.set_xlabel('Price Increase Discussed?')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/figures/07_price_increase.png', dpi=150)
plt.show()

## Plot 8: Churn Over Time (Monthly Trend)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Parse dates if not already
renewal['call_date'] = pd.to_datetime(renewal['call_date'], errors='coerce')
renewal['call_month'] = renewal['call_date'].dt.to_period('M')

# Monthly churn count
monthly_churn = renewal[renewal['is_churned'] == 'Churned'].groupby('call_month').size()
monthly_churn.plot(kind='line', ax=ax, marker='o', color='#e74c3c', linewidth=2)

ax.set_title('Monthly Churn Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Churn Events')

plt.tight_layout()
plt.savefig('../reports/figures/08_churn_trend.png', dpi=150)
plt.show()

## Plot 9: Call Frequency per Customer

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

calls_per_customer = renewal.groupby('co_ref').size()
calls_per_customer.plot(kind='hist', bins=50, ax=ax, color='#3498db', edgecolor='black')

ax.set_title('Distribution of Calls per Customer', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Calls')
ax.set_ylabel('Number of Customers')
ax.axvline(calls_per_customer.mean(), color='red', linestyle='--', label=f'Mean: {calls_per_customer.mean():.1f}')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/09_calls_per_customer.png', dpi=150)
plt.show()

## Plot 10: Top Mentioned Competitors

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Filter to rows where competitors were mentioned
competitors = renewal[renewal['mentioned_competitors'].notna()]['mentioned_competitors']
if len(competitors) > 0:
    competitors.value_counts().head(10).plot(kind='barh', ax=ax, color='#e67e22', edgecolor='black')
    ax.set_title('Top 10 Mentioned Competitors', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Mentions')
else:
    ax.text(0.5, 0.5, 'No competitor data available', ha='center', fontsize=14)

plt.tight_layout()
plt.savefig('../reports/figures/10_top_competitors.png', dpi=150)
plt.show()

## TODO: Add more plots for billing, cc_calls, and emails datasets

Suggestions:
- Billing amount distribution
- CC call frequency vs churn
- Email communication patterns

In [ ]:
# TODO: Add billing EDA plots

## EDA Key Findings Summary

### TODO: Write your key findings

1. **Churn Rate:** ...% of customers churned
2. **Top Churn Reasons:** ...
3. **Competitor Impact:** ...
4. **Discount Effectiveness:** ...
5. **Price Sensitivity:** ...
6. **Temporal Patterns:** ...